In [0]:
%run ./00_config_setup

#### 1. S3 bucket path

In [0]:
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {GOLD_SCHEMA}")


DataFrame[]

#### 2. unity catalog path

#### 3. run identity

/home/spark-7c303f03-9b34-4e8c-89b0-27/.ipykernel/1800/command-8628948029972694-2015145426:6: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  RUN_TS         = datetime.utcnow()


In [0]:
# %sql
# TRUNCATE TABLE gold.dim_store;
# TRUNCATE TABLE gold.dim_customer;

#### All Utilities

In [0]:

def upsert_dim(source_df: DataFrame, table_name: str,
               merge_key: str, domain_label: str = ""):
    
    try:
        target = DeltaTable.forName(spark, table_name)
        (
            target.alias("t")
            .merge(source_df.alias("s"), merge_key)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
        print(f"[MERGE] {table_name}")
    
    except:
        # Table not initialized — first time, just overwrite
        (
            source_df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(table_name)
        )
        print(f"[OVERWRITE] {table_name} — first load")

    cnt = spark.table(table_name).count()
    print(f"[COUNT] {table_name} → {cnt:,} rows")
    return cnt


def overwrite_fact(df: DataFrame, table_name: str, s3_path: str):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .option("path", s3_path)
        .saveAsTable(table_name)
    )
    cnt = spark.table(table_name).count()
    return cnt


def optimize_zorder(table_name: str, cols: list):
    zorder = ", ".join(cols)
    spark.sql(f"OPTIMIZE {table_name} ZORDER BY ({zorder})")


def show_detail(table_name: str):
    spark.sql(f"DESCRIBE DETAIL {table_name}").select(
        "name", "location", "numFiles", "sizeInBytes"
    ).show(truncate=False)


#### DIM_DATE

In [0]:
# DIM_DATE — Static calendar spine 2010-01-01 to 2025-12-31
# Covers all three domains:
#   Electronics : 2019 , Liquor      : 2012-2020, Books       : 1995-2023 (reviews) — we extend to 2025 for safety

start = dt_date(1990, 1, 1) 
end   = dt_date(2025, 12, 31)


date_rows = []
current = start
day_names  = ["Sunday","Monday","Tuesday","Wednesday","Thursday","Friday","Saturday"]
month_names = ["","January","February","March","April","May","June",
               "July","August","September","October","November","December"]

while current <= end:
    dow      = current.isoweekday() % 7   # isoweekday: Mon=1…Sun=7 → convert to 1=Sun…7=Sat
    dow_spark = dow + 1 if dow < 7 else 1  # Spark: 1=Sun, 7=Sat
    is_weekend = dow_spark in (1, 7)

    # Last day of month check
    next_day = current + timedelta(days=1)
    is_month_end = (next_day.month != current.month)

    date_rows.append((
        int(current.strftime("%Y%m%d")),    # date_key
        current,                             # full_date
        current.year,                        # year
        current.month,                       # month
        month_names[current.month],          # month_name
        (current.month - 1) // 3 + 1,       # quarter
        f"Q{(current.month - 1) // 3 + 1}", # quarter_name
        int(current.strftime("%W")),         # week_of_year
        dow_spark,                           # day_of_week
        day_names[dow_spark - 1],            # day_name
        is_weekend,                          # is_weekend
        is_month_end                         # is_month_end
    ))
    current += timedelta(days=1)

schema_date = [
    "date_key","full_date","year","month","month_name",
    "quarter","quarter_name","week_of_year","day_of_week",
    "day_name","is_weekend","is_month_end"
]

df_date = spark.createDataFrame(date_rows, schema=schema_date)
df_date = df_date.withColumn("full_date", col("full_date").cast(DateType()))

#ensure_delta_ready(TBL_DIM_DATE,      S3_GOLD_DIM_DATE,      df_date)

upsert_dim(df_date, TBL_DIM_DATE, "t.date_key = s.date_key", "calendar" )
optimize_zorder(TBL_DIM_DATE, ["date_key"])
print("\nSample DIM_DATE:")
display(spark.table(TBL_DIM_DATE).limit(5))

[MERGE] retailflow360.gold.dim_date
[COUNT] retailflow360.gold.dim_date → 13,149 rows

Sample DIM_DATE:


date_key,full_date,year,month,month_name,quarter,quarter_name,week_of_year,day_of_week,day_name,is_weekend,is_month_end
20161231,2016-12-31,2016,12,December,4,Q4,52,7,Saturday,true,true
20170101,2017-01-01,2017,1,January,1,Q1,0,1,Sunday,true,false
20170102,2017-01-02,2017,1,January,1,Q1,1,2,Monday,false,false
20170103,2017-01-03,2017,1,January,1,Q1,1,3,Tuesday,false,false
20170104,2017-01-04,2017,1,January,1,Q1,1,4,Wednesday,false,false


#### DIM_DOMAIN

In [0]:
# DIM_DOMAIN — 3 static rows, one per source domain
# MERGE on domain_key — idempotent

domain_rows = [
    ("Electronics", "Electronics e-commerce sales — 12 monthly US city order files, 2019"),
    ("Liquor",      "Iowa State Liquor Sales — retail transactions across Iowa counties, 2012–2020"),
    ("Books",       "Amazon book catalog and user reviews — metadata and ratings data"),
]

df_domain = spark.createDataFrame(domain_rows, ["domain_name", "domain_desc"])
df_domain = (
    df_domain
    .withColumn("domain_key",  md5(col("domain_name")))
    .withColumn("created_at",  current_timestamp())
)

upsert_dim(df_domain, TBL_DIM_DOMAIN, "t.domain_key = s.domain_key", "static")
display(spark.table(TBL_DIM_DOMAIN))

[OVERWRITE] retailflow360.gold.dim_domain — first load
[COUNT] retailflow360.gold.dim_domain → 3 rows


domain_name,domain_desc,domain_key,created_at
Electronics,"Electronics e-commerce sales — 12 monthly US city order files, 2019",ee3c6e8ed9c27c45f161ea416c997df8,2026-04-22T10:44:57.166Z
Liquor,"Iowa State Liquor Sales — retail transactions across Iowa counties, 2012–2020",86045fb199c107f5a7ba7c90772e0491,2026-04-22T10:44:57.166Z
Books,Amazon book catalog and user reviews — metadata and ratings data,6225eb5bf8a031f750a1b03f810ccc6a,2026-04-22T10:44:57.166Z


#### DIM_PRODUCT

In [0]:
# ═══════════════════════════════════════════════════════════════
# DIM_PRODUCT — SCD Type 1 (overwrite on change)
# Sources:
#   Electronics : product name + price_each
#   Liquor      : item_description + category_name + state_bottle_retail
#   Books       : title + primary_category + price
# Surrogate key: md5(product_name + domain)
# Natural key  : product_name + domain
# MERGE: update unit_price and category if changed (SCD1 — no history)
# ═══════════════════════════════════════════════════════════════

# Electronics products
df_elec_slv = spark.read.format("delta").load(S3_SILVER_ELECTRONICS)
df_prod_elec = (
    df_elec_slv
    .filter(col("row_quality_flag") != "CRITICAL_NULL_PK")
    .select(
        col("product").alias("product_name"),
        col("price_each").alias("unit_price")
    )
    .withColumn("category",   lit("Electronics"))
    .withColumn("domain",     lit("Electronics"))
    .dropDuplicates(["product_name", "domain"])
)

# Liquor products
df_liq_slv = spark.read.format("delta").load(S3_SILVER_LIQUOR)
df_prod_liq = (
    df_liq_slv
    .filter(col("row_quality_flag") != "CRITICAL_NULL_PK")
    .select(
        col("item_description").alias("product_name"),
        coalesce(col("category_name"), lit("Liquor")).alias("category"),
        col("state_bottle_retail").alias("unit_price")
    )
    .withColumn("domain", lit("Liquor"))
    .dropDuplicates(["product_name", "domain"])
)

# Books products
df_bd_slv  = spark.read.format("delta").load(S3_SILVER_BOOKS_DATA)
df_br_slv  = spark.read.format("delta").load(S3_SILVER_BOOKS_RATING)

# Use books_data as primary source for book catalog
df_prod_books = (
    df_bd_slv
    .filter(col("title").isNotNull())
    .select(
        col("title").alias("product_name"),
        coalesce(col("primary_category"), lit("Uncategorized")).alias("category"),
        lit(None).cast(DoubleType()).alias("unit_price")
    )
    .withColumn("domain", lit("Books"))
    .dropDuplicates(["product_name", "domain"])
)

# Enrich books unit_price from reviews (price column)
df_book_prices = (
    df_br_slv
    .filter(col("price").isNotNull())
    .groupBy("title")
    .agg(F.avg("price").alias("avg_price"))
)

df_prod_books = (
    df_prod_books
    .join(df_book_prices, df_prod_books["product_name"] == df_book_prices["title"], "left")
    .withColumn("unit_price", coalesce(col("unit_price"), col("avg_price")))
    .drop("title", "avg_price")
)

# Union all three domains
df_product = (
    df_prod_elec
    .unionByName(df_prod_liq, allowMissingColumns=True)
    .unionByName(df_prod_books, allowMissingColumns=True)
)

# Add surrogate key and audit columns
df_product = (
    df_product
    .filter(col("product_name").isNotNull())
    .withColumn("product_key",
        md5(concat_ws("|", col("product_name"), col("domain"))))
    .withColumn("updated_at", current_timestamp())
    .dropDuplicates(["product_key"])
)

print(f"  Electronics products : {df_prod_elec.count():,}")
print(f"  Liquor products      : {df_prod_liq.count():,}")
print(f"  Books products       : {df_prod_books.count():,}")
print(f"  Total DIM_PRODUCT    : {df_product.count():,}")

upsert_dim(
    df_product, TBL_DIM_PRODUCT,
    "t.product_key = s.product_key",
    "SCD1"
)
optimize_zorder(TBL_DIM_PRODUCT, ["product_key", "domain"])
print("\nSample DIM_PRODUCT:")
display(spark.table(TBL_DIM_PRODUCT).limit(5))

#### DIM_GEOGRAPHY 

In [0]:
# ═══════════════════════════════════════════════════════════════
# DIM_GEOGRAPHY — SCD Type 1
# Sources: Electronics (city+state+zip) + Liquor (city+county+zip+lat+lon)
# Surrogate key: md5(city + state + zip_code)
# Electronics has state; Liquor has county+lat+lon — union both
# MERGE on geo_key

# Electronics geography
df_geo_elec = (
    df_elec_slv
    .filter(
        col("city").isNotNull() &
        col("state").isNotNull()
    )
    .select(
        col("city"),
        col("state"),
        coalesce(col("zip_code"), lit("")).alias("zip_code"),
        lit(None).cast(StringType()).alias("county"),
        lit(None).cast(DoubleType()).alias("lat"),
        lit(None).cast(DoubleType()).alias("lon"),
        lit("Electronics").alias("domain")
    )
    .dropDuplicates(["city", "state", "zip_code"])
)

# Liquor geography
df_geo_liq = (
    df_liq_slv
    .filter(col("city").isNotNull())
    .select(
        col("city"),
        lit("IA").alias("state"),             # Iowa — all liquor data is Iowa
        coalesce(col("zip_code"), lit("")).alias("zip_code"),
        col("county"),
        col("lat"),
        col("lon"),
        lit("Liquor").alias("domain")
    )
    .dropDuplicates(["city", "state", "zip_code"])
)

df_geography = (
    df_geo_elec
    .unionByName(df_geo_liq, allowMissingColumns=True)
    .withColumn("geo_key",
        md5(concat_ws("|",
            coalesce(col("city"), lit("")),
            coalesce(col("state"), lit("")),
            coalesce(col("zip_code"), lit(""))
        ))
    )
    .withColumn("updated_at", current_timestamp())
    .dropDuplicates(["geo_key"])
)

print(f"  Electronics locations : {df_geo_elec.count():,}")
print(f"  Liquor locations      : {df_geo_liq.count():,}")
print(f"  Total DIM_GEOGRAPHY   : {df_geography.count():,}")

upsert_dim(
    df_geography, TBL_DIM_GEOGRAPHY,
    "t.geo_key = s.geo_key",
    "SCD1"
)
optimize_zorder(TBL_DIM_GEOGRAPHY, ["geo_key", "city", "state"])
print("\nSample DIM_GEOGRAPHY:")
display(spark.table(TBL_DIM_GEOGRAPHY).limit(5))

#### DIM_STORE

In [0]:
# ═══════════════════════════════════════════════════════════════
# DIM_STORE — SCD Type 2 (Liquor domain only)
# Electronics and Books: no store concept → store_key = null in fact
# Natural key: store_number
# SCD2 logic: snapshot approach
#   1. Read current Silver store data
#   2. Read current DIM_STORE active rows (is_current = true)
#   3. Find stores where store_name or address changed
#   4. Close old rows (expiry_date = today, is_current = false)
#   5. Insert new rows (effective_date = today, expiry_date = 9999-12-31)
# Surrogate key: md5(store_number + effective_date)
# ═══════════════════════════════════════════════════════════════

print("=" * 60)
print("DIM_STORE (SCD2)")
print("=" * 60)

# Current state of stores from Silver
df_stores_new = (
    df_liq_slv
    .filter(
        col("store_number").isNotNull() &
        (col("row_quality_flag") != "CRITICAL_NULL_PK")
    )
    .select(
        col("store_number"),
        col("store_name"),
        col("address"),
        col("city"),
        col("zip_code"),
        col("county")
    )
    .dropDuplicates(["store_number"])
)

store_count_silver = df_stores_new.count()
print(f"  Unique stores in Silver: {store_count_silver:,}")

# Check if DIM_STORE already has data
try:
    existing_stores = spark.table(TBL_DIM_STORE).filter(col("is_current") == True)
    existing_count  = existing_stores.count()
except Exception:
    existing_count = 0

print(f"  Existing active rows in DIM_STORE: {existing_count:,}")

if existing_count == 0:
    # ── First run: insert all stores as current ─────────────────
    df_store_insert = (
        df_stores_new
        .withColumn("effective_date", to_date(lit(RUN_DATE)))
        .withColumn("expiry_date",    to_date(lit(SCD2_MAX_DATE)))
        .withColumn("is_current",     lit(True).cast(BooleanType()))
        .withColumn("store_key",
            md5(concat_ws("|",
                col("store_number").cast(StringType()),
                col("effective_date").cast(StringType())
            ))
        )
        .withColumn("created_at", current_timestamp())
    )

    (
        df_store_insert.write
        .format("delta")
        .mode("overwrite")
        .option("path", S3_GOLD_DIM_STORE)
        .saveAsTable(TBL_DIM_STORE)
    )
    print(f"  ✅ First load: {df_store_insert.count():,} stores inserted")

else:
    # ── Subsequent runs: SCD2 — detect changes, close old, insert new ──
    # Join new snapshot to existing active rows
    df_changed = (
        df_stores_new.alias("new")
        .join(
            existing_stores.select(
                "store_number", "store_name", "address", "store_key"
            ).alias("old"),
            "store_number",
            "left"
        )
        .filter(
            # Store is new (no existing row) OR attributes changed
            col("old.store_key").isNull() |
            (col("new.store_name") != col("old.store_name")) |
            (col("new.address")    != col("old.address"))
        )
        .select("new.*")
    )

    changed_count = df_changed.count()
    print(f"  Changed/new stores detected: {changed_count:,}")

    if changed_count > 0:
        # Close old active rows for changed stores
        (
            DeltaTable.forName(spark, TBL_DIM_STORE)
            .alias("t")
            .merge(
                df_changed.select("store_number").alias("s"),
                "t.store_number = s.store_number AND t.is_current = true"
            )
            .whenMatchedUpdate(set={
                "expiry_date": f"to_date('{RUN_DATE}')",
                "is_current":  "false"
            })
            .execute()
        )

        # Insert new/updated rows
        df_new_rows = (
            df_changed
            .withColumn("effective_date", to_date(lit(RUN_DATE)))
            .withColumn("expiry_date",    to_date(lit(SCD2_MAX_DATE)))
            .withColumn("is_current",     lit(True).cast(BooleanType()))
            .withColumn("store_key",
                md5(concat_ws("|",
                    col("store_number").cast(StringType()),
                    col("effective_date").cast(StringType())
                ))
            )
            .withColumn("created_at", current_timestamp())
        )

        (
            df_new_rows.write
            .format("delta")
            .mode("append")
            .option("path", S3_GOLD_DIM_STORE)
            .saveAsTable(TBL_DIM_STORE)
        )
        print(f"  ✅ SCD2: closed old rows + inserted {changed_count:,} new rows")
    else:
        print(f"  ✅ No store changes detected — DIM_STORE unchanged")

dim_store_total = spark.table(TBL_DIM_STORE).count()
dim_store_active = spark.table(TBL_DIM_STORE).filter(col("is_current") == True).count()
print(f"  DIM_STORE total rows : {dim_store_total:,}")
print(f"  DIM_STORE active rows: {dim_store_active:,}")

optimize_zorder(TBL_DIM_STORE, ["store_number", "is_current"])
print("\nSample DIM_STORE:")
display(spark.table(TBL_DIM_STORE).limit(5))

#### DIM_CUSTOMER

In [0]:
# ═══════════════════════════════════════════════════════════════
# DIM_CUSTOMER — SCD Type 2 (Books domain only)
# Book reviewers as proxy for customer
# Natural key: user_id
# Electronics/Liquor: no customer → customer_key = null in fact
# SCD2: same snapshot approach as DIM_STORE
# Note: ANONYMOUS users all share one customer row (is_anonymous=true)
# ═══════════════════════════════════════════════════════════════

df_customers_new = (
    df_br_slv
    .filter(col("user_id").isNotNull())
    .select(
        col("user_id"),
        col("profilename").alias("profile_name")
    )
    .groupBy("user_id")
    .agg(F.first("profile_name", ignorenulls=True).alias("profile_name"))
    .withColumn("is_anonymous",
        (col("user_id") == "ANONYMOUS").cast(BooleanType()))
)

customer_count = df_customers_new.count()
print(f"  Unique customers in Silver: {customer_count:,}")

try:
    existing_customers = spark.table(TBL_DIM_CUSTOMER).filter(col("is_current") == True)
    existing_cust_count = existing_customers.count()
except Exception:
    existing_cust_count = 0

if existing_cust_count == 0:
    # First run
    df_cust_insert = (
        df_customers_new
        .withColumn("effective_date", to_date(lit(RUN_DATE)))
        .withColumn("expiry_date",    to_date(lit(SCD2_MAX_DATE)))
        .withColumn("is_current",     lit(True).cast(BooleanType()))
        .withColumn("customer_key",   md5(col("user_id")))
        .withColumn("created_at",     current_timestamp())
    )

    (
        df_cust_insert.write
        .format("delta")
        .mode("overwrite")
        .option("path", S3_GOLD_DIM_CUSTOMER)
        .saveAsTable(TBL_DIM_CUSTOMER)
    )
    print(f"  ✅ First load: {df_cust_insert.count():,} customers inserted")

else:
    # Subsequent runs: detect profile_name changes
    df_cust_changed = (
        df_customers_new.alias("new")
        .join(
            existing_customers.select(
                "user_id", "profile_name", "customer_key"
            ).alias("old"),
            "user_id", "left"
        )
        .filter(
            col("old.customer_key").isNull() |
            (
                col("new.profile_name").isNotNull() &
                col("old.profile_name").isNotNull() &
                (col("new.profile_name") != col("old.profile_name"))
            )
        )
        .select("new.*")
    )

    changed_cust = df_cust_changed.count()
    print(f"  Changed/new customers: {changed_cust:,}")

    if changed_cust > 0:
        # Close old rows
        (
            DeltaTable.forName(spark, TBL_DIM_CUSTOMER)
            .alias("t")
            .merge(
                df_cust_changed.select("user_id").alias("s"),
                "t.user_id = s.user_id AND t.is_current = true"
            )
            .whenMatchedUpdate(set={
                "expiry_date": f"to_date('{RUN_DATE}')",
                "is_current":  "false"
            })
            .execute()
        )

        # Insert new rows
        df_cust_new_rows = (
            df_cust_changed
            .withColumn("effective_date", to_date(lit(RUN_DATE)))
            .withColumn("expiry_date",    to_date(lit(SCD2_MAX_DATE)))
            .withColumn("is_current",     lit(True).cast(BooleanType()))
            .withColumn("customer_key",   md5(col("user_id")))
            .withColumn("created_at",     current_timestamp())
        )

        (
            df_cust_new_rows.write
            .format("delta")
            .mode("append")
            .option("path", S3_GOLD_DIM_CUSTOMER)
            .saveAsTable(TBL_DIM_CUSTOMER)
        )

dim_cust_total  = spark.table(TBL_DIM_CUSTOMER).count()
dim_cust_active = spark.table(TBL_DIM_CUSTOMER).filter(col("is_current") == True).count()
print(f"  DIM_CUSTOMER total rows : {dim_cust_total:,}")
print(f"  DIM_CUSTOMER active rows: {dim_cust_active:,}")

optimize_zorder(TBL_DIM_CUSTOMER, ["user_id", "is_current"])
print("\nSample DIM_CUSTOMER:")
display(spark.table(TBL_DIM_CUSTOMER).limit(5))

#### FACT_SALES

In [0]:
# ─────────────────────────────────────────────────────────────
# FIX 2: NULL line_revenue for books → coalesce(price, 0.0)
# FIX 3: Filter bottles_sold <= 0 from Liquor
# FIX 4: Deduplicate on sales_key after union
# FIX 5: NULL product_key → use fallback "UNKNOWN" product key
# ─────────────────────────────────────────────────────────────

print("=" * 60)
print("REBUILDING FACT_SALES WITH ALL FIXES")
print("=" * 60)

# ── Load Silver tables ────────────────────────────────────────
df_elec_slv = spark.read.format("delta").load(f"s3://{BUCKET}/silver/electronics")
df_liq_slv  = spark.read.format("delta").load(f"s3://{BUCKET}/silver/liquor_sales")
df_bd_slv   = spark.read.format("delta").load(f"s3://{BUCKET}/silver/books_data")
df_br_slv   = spark.read.format("delta").load(f"s3://{BUCKET}/silver/books_rating")

# ── Load dimension lookups ─────────────────────────────────────
df_dim_date    = spark.table(TBL_DIM_DATE).select("date_key", "full_date")
df_dim_product = spark.table(TBL_DIM_PRODUCT).select("product_key", "product_name", "domain")
df_dim_geo     = spark.table(TBL_DIM_GEOGRAPHY).select("geo_key", "city", "state", "zip_code", "domain")
df_dim_domain  = spark.table(TBL_DIM_DOMAIN).select("domain_key", "domain_name")


# Get domain keys once
electronics_dk = df_dim_domain.filter(col("domain_name") == "Electronics").select("domain_key").first()["domain_key"]
liquor_dk      = df_dim_domain.filter(col("domain_name") == "Liquor").select("domain_key").first()["domain_key"]
books_dk       = df_dim_domain.filter(col("domain_name") == "Books").select("domain_key").first()["domain_key"]

# FIX 5: create a fallback "UNKNOWN_PRODUCT" key for unresolved joins
UNKNOWN_PRODUCT_KEY = md5(lit("UNKNOWN|UNKNOWN")).alias("unknown_key")
unknown_product_key_val = spark.range(1).select(
    F.md5(F.lit("UNKNOWN|UNKNOWN")).alias("k")
).first()["k"]

print(f"  Domain keys loaded: Electronics={electronics_dk[:8]}..., "
      f"Liquor={liquor_dk[:8]}..., Books={books_dk[:8]}...")




print("\n── Electronics ──")

df_elec_clean = (
    df_elec_slv
    .filter(
        (col("row_quality_flag") != "CRITICAL_NULL_PK") &
        col("order_date_dt").isNotNull() &
        col("product").isNotNull() &
        (~col("is_cross_year"))                    # only clean 2019 data
    )
)

# Date key
df_fe = df_elec_clean.withColumn(
    "date_key",
    (year(col("order_date_dt")) * 10000 +
     month(col("order_date_dt")) * 100 +
     F.dayofmonth(col("order_date_dt"))).cast(IntegerType())
)

# Product key
df_prod_elec = (
    df_dim_product.filter(col("domain") == "Electronics")
    .select(col("product_key").alias("_pk"), col("product_name").alias("_pn"))
)
df_fe = df_fe.join(F.broadcast(df_prod_elec), col("product") == col("_pn"), "left")

# FIX 5: coalesce product_key to unknown fallback
df_fe = df_fe.withColumn(
    "product_key",
    coalesce(col("_pk"), lit(unknown_product_key_val))
)

# Geo key
df_geo_elec = (
    df_dim_geo.filter(col("domain") == "Electronics")
    .select(col("geo_key").alias("_gk"), "city", "state", "zip_code")
)
df_fe = (
    df_fe
    .withColumn("_gk_lookup",
        F.md5(F.concat_ws("|",
            F.coalesce(col("city"), lit("")),
            F.coalesce(col("state"), lit("")),
            F.coalesce(col("zip_code"), lit(""))
        ))
    )
    .join(F.broadcast(df_geo_elec.select(col("_gk"), col("city").alias("_gc"))),
          col("_gk_lookup") == col("_gk"), "left")
)

# Build fact rows
df_fact_elec = (
    df_fe
    .withColumn("sales_key",
        F.md5(F.concat_ws("|", col("order_id"), col("product"),
                           col("order_date_dt").cast(StringType())))
    )
    .select(
        col("sales_key"),
        col("product_key"),
        lit(None).cast(StringType()).alias("store_key"),
        lit(None).cast(StringType()).alias("customer_key"),
        col("date_key"),
        col("_gk").alias("geo_key"),
        lit(electronics_dk).alias("domain_key"),
        lit("Electronics").alias("domain"),
        col("quantity_ordered").cast(IntegerType()).alias("quantity"),
        col("price_each").alias("unit_price"),
        col("line_revenue"),                        # already calculated in Silver
        lit(None).cast(DoubleType()).alias("review_score"),
        lit(None).cast(IntegerType()).alias("helpful_votes"),
        lit(None).cast(IntegerType()).alias("total_votes"),
        col("source_file_name"),
        col("batch_id"),
        current_timestamp().alias("gold_timestamp")
    )
    .filter(col("sales_key").isNotNull())
)

elec_count = df_fact_elec.count()
print(f"  Electronics rows: {elec_count:,}")
null_dk_elec = df_fact_elec.filter(col("date_key").isNull()).count()
null_pk_elec = df_fact_elec.filter(col("product_key").isNull()).count()
print(f"  NULL date_key: {null_dk_elec:,}  |  NULL product_key: {null_pk_elec:,}")

In [0]:
print("\n── Liquor ──")

df_liq_clean = (
    df_liq_slv
    .filter(
        (col("row_quality_flag") != "CRITICAL_NULL_PK") &
        col("transaction_date").isNotNull() &
        col("item_description").isNotNull() &
        # FIX 3: filter out zero/negative bottles_sold
        col("bottles_sold").isNotNull() &
        (col("bottles_sold") > 0)
    )
)

# Date key
df_fl = df_liq_clean.withColumn(
    "date_key",
    (year(col("transaction_date")) * 10000 +
     month(col("transaction_date")) * 100 +
     F.dayofmonth(col("transaction_date"))).cast(IntegerType())
)

# Product key
df_prod_liq = (
    df_dim_product.filter(col("domain") == "Liquor")
    .select(col("product_key").alias("_pk"), col("product_name").alias("_pn"))
)
df_fl = df_fl.join(F.broadcast(df_prod_liq),
                    col("item_description") == col("_pn"), "left")
df_fl = df_fl.withColumn(
    "product_key",
    coalesce(col("_pk"), lit(unknown_product_key_val))
)

# Geo key
df_geo_liq = (
    df_dim_geo.filter(col("domain") == "Liquor")
    .select(col("geo_key").alias("_gk"), "city", "state", "zip_code")
)
df_fl = (
    df_fl
    .withColumn("_gk_lookup",
        F.md5(F.concat_ws("|",
            F.coalesce(col("city"), lit("")),
            lit("IA"),
            F.coalesce(col("zip_code"), lit(""))
        ))
    )
    .join(F.broadcast(df_geo_liq.select("_gk")),
          col("_gk_lookup") == col("_gk"), "left")
)

# Store key
df_store_active = (
    spark.table(TBL_DIM_STORE)
    .filter(col("is_current") == True)
    .select(col("store_key").alias("_sk"), col("store_number").alias("_sn"))
)
df_fl = df_fl.join(F.broadcast(df_store_active),
                    col("store_number") == col("_sn"), "left")

# Build Liquor fact rows
df_fact_liq = (
    df_fl
    .withColumn("sales_key",
        F.md5(F.concat_ws("|", col("invoice_item_number")))
    )
    .select(
        col("sales_key"),
        col("product_key"),
        col("_sk").alias("store_key"),
        lit(None).cast(StringType()).alias("customer_key"),
        col("date_key"),
        col("_gk").alias("geo_key"),
        lit(liquor_dk).alias("domain_key"),
        lit("Liquor").alias("domain"),
        col("bottles_sold").cast(IntegerType()).alias("quantity"),
        col("state_bottle_retail").alias("unit_price"),
        col("sale__dollars_").alias("line_revenue"),
        lit(None).cast(DoubleType()).alias("review_score"),
        lit(None).cast(IntegerType()).alias("helpful_votes"),
        lit(None).cast(IntegerType()).alias("total_votes"),
        col("source_file_name"),
        col("batch_id"),
        current_timestamp().alias("gold_timestamp")
    )
    .filter(col("sales_key").isNotNull())
)

liq_count = df_fact_liq.count()
print(f"  Liquor rows       : {liq_count:,}")
null_dk_liq = df_fact_liq.filter(col("date_key").isNull()).count()
print(f"  NULL date_key     : {null_dk_liq:,}  (expect 0 — all dates 2012-2020 in DIM_DATE)")
null_rv_liq = df_fact_liq.filter(col("line_revenue").isNull()).count()
print(f"  NULL line_revenue : {null_rv_liq:,}  (expect near 0 — only from null sale__dollars_)")

In [0]:
print("\n── Books ──")

df_br_clean = (
    df_br_slv
    .filter(
        col("review_id").isNotNull() &
        col("book_id").isNotNull() &
        col("review_date").isNotNull() &
        col("is_valid_score") &
        col("is_valid_review_date")     # only valid dates 1995-2023
    )
)

# Date key — now works for 1995+ because DIM_DATE extended to 1990
df_fb = df_br_clean.withColumn(
    "date_key",
    (year(col("review_date")) * 10000 +
     month(col("review_date")) * 100 +
     F.dayofmonth(col("review_date"))).cast(IntegerType())
)

# Product key — join on title
df_prod_books = (
    df_dim_product.filter(col("domain") == "Books")
    .select(col("product_key").alias("_pk"), col("product_name").alias("_pn"))
)
df_fb = df_fb.join(F.broadcast(df_prod_books),
                    col("title") == col("_pn"), "left")

# FIX 5: fallback for 208 unmatched book titles
df_fb = df_fb.withColumn(
    "product_key",
    coalesce(col("_pk"), lit(unknown_product_key_val))
)

# Customer key
df_cust_active = (
    spark.table(TBL_DIM_CUSTOMER)
    .filter(col("is_current") == True)
    .select(col("customer_key").alias("_ck"), col("user_id").alias("_uid"))
)
df_fb = df_fb.join(F.broadcast(df_cust_active),
                    col("user_id") == col("_uid"), "left")

# FIX 2: line_revenue — use price if available, else 0.0 (not null)
df_fact_books = (
    df_fb
    .withColumn("sales_key",
        F.md5(F.concat_ws("|", col("review_id")))
    )
    .select(
        col("sales_key"),
        col("product_key"),
        lit(None).cast(StringType()).alias("store_key"),
        col("_ck").alias("customer_key"),
        col("date_key"),
        lit(None).cast(StringType()).alias("geo_key"),
        lit(books_dk).alias("domain_key"),
        lit("Books").alias("domain"),
        lit(1).cast(IntegerType()).alias("quantity"),
        col("price").alias("unit_price"),
        # FIX 2: coalesce price to 0.0 — never null in fact table
        coalesce(col("price"), lit(0.0)).alias("line_revenue"),
        col("review_score"),
        col("helpful_votes"),
        col("total_votes"),
        col("source_file_name"),
        col("batch_id"),
        current_timestamp().alias("gold_timestamp")
    )
    .filter(col("sales_key").isNotNull())
)

books_count = df_fact_books.count()
print(f"  Books rows        : {books_count:,}")
null_dk_books = df_fact_books.filter(col("date_key").isNull()).count()
null_rv_books = df_fact_books.filter(col("line_revenue").isNull()).count()
null_pk_books = df_fact_books.filter(col("product_key").isNull()).count()
print(f"  NULL date_key     : {null_dk_books:,}  (expect 0 after DIM_DATE fix)")
print(f"  NULL line_revenue : {null_rv_books:,}  (expect 0 — coalesced to 0.0)")
print(f"  NULL product_key  : {null_pk_books:,}  (expect 0 — fallback key applied)")

In [0]:
print("\n── Union + Deduplication + Overwrite ──")

df_fact_all = (
    df_fact_elec
    .unionByName(df_fact_liq,   allowMissingColumns=True)
    .unionByName(df_fact_books, allowMissingColumns=True)
)

df_fact_all = df_fact_all.withColumn(
    "line_revenue",
    coalesce(col("line_revenue"), lit(0.0))
)


total_before_dedup = df_fact_all.count()
print(f"  Rows before dedup : {total_before_dedup:,}")
print(f"    Electronics     : {elec_count:,}")
print(f"    Liquor          : {liq_count:,}")
print(f"    Books           : {books_count:,}")

# FIX 4: Deduplicate on sales_key
# Keep the row with the most complete data (non-null line_revenue first)
w_dedup = (
    Window
    .partitionBy("sales_key")
    .orderBy(
        col("line_revenue").desc(),    # prefer row with revenue
        col("product_key").asc()       # prefer row with product key
    )
)

df_fact_deduped = (
    df_fact_all
    .withColumn("_rank", F.row_number().over(w_dedup))
    .filter(col("_rank") == 1)
    .drop("_rank")
)

total_after_dedup = df_fact_deduped.count()
deduped_count = total_before_dedup - total_after_dedup
print(f"\n  Rows removed by dedup : {deduped_count:,}  (expected ~311)")
print(f"  Final FACT_SALES rows : {total_after_dedup:,}")

# Repartition by domain for balanced S3 output
df_fact_deduped = df_fact_deduped.repartition(200, col("domain"))

# OVERWRITE — full refresh
(
    df_fact_deduped.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("path", S3_GOLD_FACT_SALES)
    .saveAsTable(TBL_FACT_SALES)
)

final_fact_count = spark.table(TBL_FACT_SALES).count()
print(f"\n  ✅ FACT_SALES written: {final_fact_count:,} rows")

# OPTIMIZE + ZORDER
print("\n  Running OPTIMIZE + ZORDER ...")
spark.sql(f"OPTIMIZE {TBL_FACT_SALES} ZORDER BY (date_key, domain_key, product_key)")
print("  ✅ OPTIMIZE done")

In [0]:
print("\n" + "=" * 60)
print("REFERENTIAL INTEGRITY CHECKS — AFTER FIXES")
print("=" * 60)

df_fact = spark.table(TBL_FACT_SALES)
total   = df_fact.count()

print(f"\n  Total FACT_SALES rows: {total:,}\n")

checks = {
    "NULL product_key"    : df_fact.filter(col("product_key").isNull()).count(),
    "NULL date_key"       : df_fact.filter(col("date_key").isNull()).count(),
    "NULL domain_key"     : df_fact.filter(col("domain_key").isNull()).count(),
    "NULL line_revenue"   : df_fact.filter(col("line_revenue").isNull()).count(),
    "Negative quantity"   : df_fact.filter(col("quantity") <= 0).count(),
    "Negative revenue"    : df_fact.filter(col("line_revenue") < 0).count(),
    "Invalid review_score": df_fact.filter(
        col("review_score").isNotNull() &
        ((col("review_score") < 1.0) | (col("review_score") > 5.0))
    ).count(),
    "Duplicate sales_key" : (
        df_fact.groupBy("sales_key").count()
        .filter(col("count") > 1).count()
    )
}

all_passed = True
for name, failed in checks.items():
    status = "✅ PASS" if failed == 0 else "❌ FAIL"
    if failed > 0:
        all_passed = False
    pct = round(failed / total * 100, 4) if total > 0 else 0
    print(f"  {status}  {name:<25} : {failed:>10,} rows ({pct}%)")

print()
print(f"\n  Row breakdown by domain:")
spark.table(TBL_FACT_SALES) \
    .groupBy("domain") \
    .agg(
        F.count("*").alias("rows"),
        F.sum("line_revenue").alias("total_revenue"),
        F.countDistinct("product_key").alias("unique_products"),
        F.sum(when(col("date_key").isNull(), 1).otherwise(0)).alias("null_dates")
    ) \
    .orderBy("domain").show()

if all_passed:
    print("\n  ✅ ALL CHECKS PASSED — Gold layer ready for Power BI!")
else:
    print("\n  ❌ Some checks still failing — review output above")